# CPE Level Complete — Ineligible Combo Prediction Bias & Gate Removal Risk

## Methodology note

**Why traditional bias cannot be directly computed for gated combos:**  
The standard bias formula — `SUM(pred) / SUM(actual_conversion) - 1` — requires
outcome data. Outcomes are recorded only for auctions that resulted in a served
impression (model won the auction). Gated auctions bid `cost = 0`, so they never
serve an ad and never appear in `operativeecpm_installs_outcomes_contextual`.  
Direct bias = undefined for gated combos.

## Proxy-based risk analysis

Instead, risk is estimated via **three complementary proxies**:

| Section | Signal | Interpretation |
|---------|--------|----------------|
| 1 | Gated prediction distribution | High `avg_pred` = model is confident about unseen combos |
| 2 | Eligible bias for the same game | Model calibration quality for that game |
| 3 | Cross-event proxy (event gated on some games, eligible on others) | Actual bias when that event IS bidded on |
| 4 | Risk ranking | Combines volume × prediction magnitude × game-level calibration |

**Gating signal** (in `mz_dcpi_prediction_v1`):
```
body.app_event_p > 0   -- model has a non-zero prediction
AND body.cst = 0       -- gate zeroed the bid cost
AND body.max_cst > 0   -- active campaign with a target CPE
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

client = bigquery.Client(project='unity-ads-ds-prd')

# --- Section 1: prediction_v1alpha1 (logs ALL requests incl. gated, 30-day retention)
PRED_TABLE   = 'unity-data-ads-prd.dcpi.prediction_v1alpha1'
LC_TYPE      = 'APP_EVENT_CONVERSION_TYPE_LEVEL_COMPLETE'
# Use last 7 days (table has 30-day retention, hour-partitioned)
TS_START     = '2026-08-25 00:00:00'
TS_END       = '2026-09-01 23:59:59'

# --- Section 2: mz_dcpi_prediction_v1 (only bids, but has auction_id for outcome join)
ANALYSIS_START = '2026-06-15'
DOWNTIME_START = '2026-07-25'
DOWNTIME_END   = '2026-08-06'

print('BigQuery client initialized')
print(f'Gated data window  (prediction_v1alpha1): {TS_START} → {TS_END}')
print(f'Eligible bias window (mz_dcpi_pred_v1)  : {ANALYSIS_START} → today (excl downtime)')

def run_query(sql):
    return client.query(sql).to_dataframe()

---
## 1. Gated Combo Prediction Distribution

Characterise what the model predicts for gated combos — without joining outcomes.  
A high `avg_pred` on a gated combo means the model is confident about those unseen
users. Whether that confidence is justified depends on the same-game calibration
analysis in Section 2.

In [ ]:
# Overall gated vs eligible volume from prediction_v1alpha1
# (this table logs ALL LC requests including gated; dcpi=0 = gate zeroed the cost)
sql_volume = f'''
WITH lc AS (
  SELECT
    app_event_campaigns.p[SAFE_OFFSET(i)]                          AS pred,
    CAST(app_event_campaigns.dcpi[SAFE_OFFSET(i)] AS FLOAT64)      AS dcpi,
    CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)] AS FLOAT64)  AS max_cost
  FROM `{PRED_TABLE}`,
    UNNEST(GENERATE_ARRAY(0, ARRAY_LENGTH(app_event_campaigns.campaign_ids)-1)) AS i
  WHERE _lapio_submit_time BETWEEN '{TS_START}' AND '{TS_END}'
    AND app_event_campaigns.app_event_type[SAFE_OFFSET(i)] = '{LC_TYPE}'
    AND app_event_campaigns.p[SAFE_OFFSET(i)] > 0
    AND CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)] AS FLOAT64) > 0
)
SELECT
  (dcpi = 0)                                               AS is_gated,
  COUNT(*)                                                 AS auction_count,
  ROUND(AVG(pred) * 100, 4)                               AS avg_pred_pct,
  ROUND(APPROX_QUANTILES(pred, 100)[OFFSET(50)] * 100, 4) AS median_pred_pct,
  ROUND(APPROX_QUANTILES(pred, 100)[OFFSET(90)] * 100, 4) AS p90_pred_pct
FROM lc
GROUP BY is_gated
ORDER BY is_gated
'''

df_volume = run_query(sql_volume)
df_volume['cohort'] = df_volume['is_gated'].map({True: 'Gated (ineligible)', False: 'Eligible'})
display(df_volume[['cohort','auction_count','avg_pred_pct','median_pred_pct','p90_pred_pct']])

In [ ]:
# Per (game_id, target_event) prediction stats for GATED combos
# Source: prediction_v1alpha1 — the only table that logs gated (cst=0) auction requests
sql_gated_combos = f'''
WITH lc_gated AS (
  SELECT
    app_event_campaigns.campaign_ids[SAFE_OFFSET(i)]               AS campaign_id,
    app_event_campaigns.p[SAFE_OFFSET(i)]                          AS pred,
    CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)] AS FLOAT64)  AS max_cost,
    LOWER(app_event_campaigns.sdk_event_name[SAFE_OFFSET(i)])      AS sdk_event_name
  FROM `{PRED_TABLE}`,
    UNNEST(GENERATE_ARRAY(0, ARRAY_LENGTH(app_event_campaigns.campaign_ids)-1)) AS i
  WHERE _lapio_submit_time BETWEEN '{TS_START}' AND '{TS_END}'
    AND app_event_campaigns.app_event_type[SAFE_OFFSET(i)] = '{LC_TYPE}'
    AND CAST(app_event_campaigns.dcpi[SAFE_OFFSET(i)] AS FLOAT64) = 0   -- gated
    AND app_event_campaigns.p[SAFE_OFFSET(i)] > 0
    AND CAST(app_event_campaigns.max_cost[SAFE_OFFSET(i)] AS FLOAT64) > 0
),
camps AS (
  SELECT id AS campaign_id, game_id AS target_game_id
  FROM `unity-data-ads-core-prd.ads_dimension_data.campaigns_v3`
  WHERE app_event_conversion_type = 'LEVEL_COMPLETE' AND archived_at IS NULL
)
SELECT
  c.target_game_id,
  g.sdk_event_name                                          AS target_event,
  COUNT(*)                                                  AS gated_auction_count,
  ROUND(AVG(g.pred) * 100, 4)                               AS avg_pred_pct,
  ROUND(APPROX_QUANTILES(g.pred, 100)[OFFSET(50)] * 100, 4) AS median_pred_pct,
  ROUND(APPROX_QUANTILES(g.pred, 100)[OFFSET(10)] * 100, 4) AS p10_pred_pct,
  ROUND(APPROX_QUANTILES(g.pred, 100)[OFFSET(90)] * 100, 4) AS p90_pred_pct,
  ROUND(AVG(g.max_cost) / 1e6, 2)                           AS avg_target_cpe_usd,
  ROUND(SUM(g.pred * g.max_cost) / 1e6, 0)                  AS suppressed_spend_ub_usd
FROM lc_gated AS g
INNER JOIN camps AS c ON c.campaign_id = g.campaign_id
GROUP BY c.target_game_id, g.sdk_event_name
HAVING COUNT(*) >= 100
ORDER BY gated_auction_count DESC
'''

df_gated_combos = run_query(sql_gated_combos)
print(f'{len(df_gated_combos)} gated (game_id, event) pairs with >= 100 auctions (window: {TS_START[:10]} → {TS_END[:10]})')
display(df_gated_combos.head(30))

In [ ]:
# Prediction distribution: top 20 gated combos by volume
top20 = df_gated_combos.head(20).copy()
top20['combo'] = top20['target_game_id'].astype(str) + ' / ' + top20['target_event']
top20 = top20.sort_values('gated_auction_count', ascending=True)

fig = go.Figure()
# Error bars show p10–p90 range
fig.add_trace(go.Bar(
    y=top20['combo'],
    x=top20['avg_pred_pct'],
    name='avg pred %',
    orientation='h',
    marker_color='#e8541e',
    error_x=dict(
        type='data',
        symmetric=False,
        array=(top20['p90_pred_pct'] - top20['avg_pred_pct']).tolist(),
        arrayminus=(top20['avg_pred_pct'] - top20['p10_pred_pct']).tolist(),
        color='gray',
    ),
    customdata=top20[['gated_auction_count','suppressed_spend_ub_usd']].values,
    hovertemplate=('%{y}<br>avg pred: %{x:.3f}%<br>'
                   'auctions: %{customdata[0]:,}<br>'
                   'suppressed spend UB: $%{customdata[1]:,.0f}<extra></extra>'),
))
fig.update_layout(
    title='Gated Combo: Avg Prediction % — Top 20 by Volume (error bars = P10–P90)',
    xaxis_title='Avg Prediction % (model confidence)',
    template='plotly_white',
    height=max(500, len(top20) * 26 + 200),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 2. Same-Game Eligible Bias (Calibration Proxy)

For each game that has gated combos, compute the model bias on **eligible** (non-gated)
combos of the same game. This measures how well the model generalises to that game.

**Logic:** If the model is well-calibrated (low bias) on the eligible events it DOES
bid on for game G, it is more likely to also be well-calibrated for the gated events
of game G → lower risk to unlock the gate for that game.  
Conversely, games with high eligible bias are already miscalibrated even for seen
combos — unlocking unseen combos carries additional risk.

In [ ]:
# Game-level bias on ELIGIBLE combos (with outcome join)
_EV_MATCH = '''(SELECT COUNT(1)
               FROM UNNEST(outs.app_event_level_complete_sdk_event_name_array.list) AS ev,
                    UNNEST(c.target_events) AS tgt_ev
               WHERE LOWER(ev.element) = LOWER(tgt_ev)) > 0'''

sql_game_bias = f'''
WITH preds AS (
  SELECT
    p.body.auction_id  AS auction_id,
    p.body.app_event_p AS pred,
    p.body.cst         AS cost,
    p.body.max_cst     AS target_cpe,
    p.body.campaign_id AS campaign_id,
    DATE_DIFF(CURRENT_DATE(), p.submit_date, DAY) AS age_days
  FROM `unity-ai-data-prd.mz_dcpi_raw.mz_dcpi_prediction_v1` AS p
  WHERE p.submit_date >= \'{ANALYSIS_START}\'
    AND NOT (p.submit_date BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND p.body.app_event_p > 0
    AND p.body.cst > 0          -- eligible only
    AND p.body.app_event_type = \'level_complete\'
    AND p.body.max_cst > 0
),
camps AS (
  SELECT
    id AS campaign_id,
    game_id AS target_game_id,
    IFNULL(sdk_event_names, []) AS target_events,
    ARRAY_LENGTH(IFNULL(sdk_event_names, [])) = 0 AS is_wildcard
  FROM `unity-data-ads-core-prd.ads_dimension_data.campaigns_v3`
  WHERE app_event_conversion_type = \'LEVEL_COMPLETE\'
    AND archived_at IS NULL
),
outs AS (
  SELECT
    o.auctionId,
    o.app_event_level_complete_count_d7,
    o.app_event_level_complete_sdk_event_name_array
  FROM `unity-data-ads-core-prd.ads_secondary_conversion.operativeecpm_installs_outcomes_contextual` AS o
  WHERE DATE(o.adRequestTimestamp) >= \'{ANALYSIS_START}\'
    AND NOT (DATE(o.adRequestTimestamp) BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND o.campaignType = \'appEventConversion\'
)
SELECT
  c.target_game_id,
  COUNT(*)               AS eligible_auction_count,
  ROUND(SUM(p.cost) / 1e6, 2) AS eligible_spend_usd,
  -- Model bias for eligible combos of this game
  ROUND(100 * (
    SUM(CASE WHEN age_days > 8 AND c.campaign_id IS NOT NULL THEN p.pred END)
    / NULLIF(
        SUM(CASE WHEN age_days > 8 AND c.campaign_id IS NOT NULL
                 THEN CASE
                        WHEN outs.app_event_level_complete_count_d7 = 0 THEN 0.0
                        WHEN c.is_wildcard THEN 1.0
                        ELSE IF({_EV_MATCH}, 1.0, 0.0)
                      END
            END), 0
      ) - 1), 2) AS eligible_model_bias_pct,
  -- Product bias for eligible combos
  ROUND(100 * (
    SUM(CASE WHEN age_days >= 9 AND c.campaign_id IS NOT NULL THEN p.cost END)
    / NULLIF(
        SUM(CASE WHEN age_days >= 9 AND c.campaign_id IS NOT NULL
                 THEN CASE
                        WHEN outs.app_event_level_complete_count_d7 = 0 THEN 0.0
                        WHEN c.is_wildcard THEN p.target_cpe
                        ELSE IF({_EV_MATCH}, p.target_cpe, 0.0)
                      END
            END), 0
      ) - 1), 2) AS eligible_product_bias_pct
FROM preds AS p
INNER JOIN outs ON outs.auctionId = p.auction_id
LEFT JOIN camps AS c ON c.campaign_id = p.campaign_id
WHERE c.campaign_id IS NOT NULL
GROUP BY c.target_game_id
HAVING COUNT(*) >= 200
ORDER BY eligible_spend_usd DESC
'''

df_game_bias = run_query(sql_game_bias)
print(f'{len(df_game_bias)} games with eligible LC bias data')
display(df_game_bias.head(30))

---
## 3. Cross-Event Proxy Bias

Some event names appear **gated on certain games** but **eligible on other games**.  
For those event names, we can compute the actual model bias when they ARE eligible,
giving a direct signal for how well the model performs on that event type.

In [ ]:
# Find event names that are both gated (somewhere) and eligible (somewhere)
gated_events = set(df_gated_combos['target_event'].str.lower().unique())
print(f'Distinct gated event names: {len(gated_events)}')

# Compute bias by event name for eligible combos (same as Section 5 of bias breakdown notebook)
sql_event_bias = f'''
WITH preds AS (
  SELECT
    p.body.auction_id  AS auction_id,
    p.body.app_event_p AS pred,
    p.body.cst         AS cost,
    p.body.max_cst     AS target_cpe,
    p.body.campaign_id AS campaign_id,
    DATE_DIFF(CURRENT_DATE(), p.submit_date, DAY) AS age_days
  FROM `unity-ai-data-prd.mz_dcpi_raw.mz_dcpi_prediction_v1` AS p
  WHERE p.submit_date >= \'{ANALYSIS_START}\'
    AND NOT (p.submit_date BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND p.body.app_event_p > 0
    AND p.body.cst > 0
    AND p.body.app_event_type = \'level_complete\'
    AND p.body.max_cst > 0
),
camps AS (
  SELECT
    id AS campaign_id,
    game_id AS target_game_id,
    IFNULL(sdk_event_names, []) AS target_events,
    ARRAY_LENGTH(IFNULL(sdk_event_names, [])) = 0 AS is_wildcard
  FROM `unity-data-ads-core-prd.ads_dimension_data.campaigns_v3`
  WHERE app_event_conversion_type = \'LEVEL_COMPLETE\'
    AND archived_at IS NULL
),
outs AS (
  SELECT
    o.auctionId,
    o.app_event_level_complete_count_d7,
    o.app_event_level_complete_sdk_event_name_array
  FROM `unity-data-ads-core-prd.ads_secondary_conversion.operativeecpm_installs_outcomes_contextual` AS o
  WHERE DATE(o.adRequestTimestamp) >= \'{ANALYSIS_START}\'
    AND NOT (DATE(o.adRequestTimestamp) BETWEEN \'{DOWNTIME_START}\' AND \'{DOWNTIME_END}\')
    AND o.campaignType = \'appEventConversion\'
),
camp_events AS (
  SELECT c.campaign_id, LOWER(ev) AS target_event_name
  FROM camps AS c
  CROSS JOIN UNNEST(c.target_events) AS ev
  WHERE NOT c.is_wildcard
)
SELECT
  ce.target_event_name,
  COUNT(DISTINCT p.campaign_id)  AS campaign_count,
  COUNT(*)                        AS eligible_auction_count,
  ROUND(SUM(p.cost) / 1e6, 2)    AS eligible_spend_usd,
  ROUND(100 * (
    SUM(CASE WHEN age_days > 8 THEN p.pred END)
    / NULLIF(
        SUM(CASE WHEN age_days > 8
                 THEN IF(
                   outs.app_event_level_complete_count_d7 = 0, 0.0,
                   IF((SELECT COUNT(1)
                       FROM UNNEST(outs.app_event_level_complete_sdk_event_name_array.list) AS ev_item
                       WHERE LOWER(ev_item.element) = ce.target_event_name
                      ) > 0, 1.0, 0.0))
             END), 0) - 1), 2) AS eligible_model_bias_pct,
  ROUND(100 * (
    SUM(CASE WHEN age_days >= 9 THEN p.cost END)
    / NULLIF(
        SUM(CASE WHEN age_days >= 9
                 THEN IF(
                   outs.app_event_level_complete_count_d7 = 0, 0.0,
                   IF((SELECT COUNT(1)
                       FROM UNNEST(outs.app_event_level_complete_sdk_event_name_array.list) AS ev_item
                       WHERE LOWER(ev_item.element) = ce.target_event_name
                      ) > 0, p.target_cpe, 0.0))
             END), 0) - 1), 2) AS eligible_product_bias_pct
FROM preds AS p
INNER JOIN outs ON outs.auctionId = p.auction_id
INNER JOIN camp_events AS ce ON ce.campaign_id = p.campaign_id
GROUP BY ce.target_event_name
HAVING COUNT(*) >= 100
ORDER BY eligible_spend_usd DESC
'''

df_event_bias = run_query(sql_event_bias)

# Flag which event names also appear as gated on other games
df_event_bias['also_gated_elsewhere'] = df_event_bias['target_event_name'].isin(gated_events)
df_cross = df_event_bias[df_event_bias['also_gated_elsewhere']].copy()

print(f'Event names with eligible bias data: {len(df_event_bias)}')
print(f'Of which also appear as gated on other games: {len(df_cross)}')
display(df_cross.sort_values('eligible_spend_usd', ascending=False).head(30))

In [ ]:
df_cv = df_cross.dropna(subset=['eligible_model_bias_pct']).copy()
df_cv = df_cv.sort_values('eligible_spend_usd', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_cv['target_event_name'],
    x=df_cv['eligible_model_bias_pct'],
    name='Eligible model bias (same event, other games)',
    orientation='h',
    marker_color='#7b4fa8', opacity=0.85,
    customdata=df_cv[['eligible_auction_count','eligible_spend_usd']].values,
    hovertemplate=('%{y}<br>eligible model bias: %{x:.1f}%<br>'
                   'eligible auctions: %{customdata[0]:,}<br>'
                   'eligible spend: $%{customdata[1]:,.0f}<extra></extra>'),
))
fig.add_trace(go.Bar(
    y=df_cv['target_event_name'],
    x=df_cv['eligible_product_bias_pct'],
    name='Eligible product bias',
    orientation='h',
    marker_color='#e8a020', opacity=0.75,
    hovertemplate='%{y}<br>eligible product bias: %{x:.1f}%<extra></extra>',
))

fig.add_vline(x=20,  line_dash='dot', line_color='lightgray', annotation_text='+20%')
fig.add_vline(x=-20, line_dash='dot', line_color='lightgray', annotation_text='-20%')
fig.add_vline(x=0,   line_color='black', line_width=1)

fig.update_layout(
    title='Cross-Event Proxy: Eligible Bias for Event Names That Are Also Gated Elsewhere',
    xaxis_title='Bias % (from eligible combos where this event IS bidded)',
    barmode='group',
    template='plotly_white',
    height=max(500, len(df_cv) * 28 + 200),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

---
## 4. Risk Ranking — Gate Removal Candidates

Combines the three proxy signals into a risk tier for each gated combo:

```
proxy_risk_score = log10(gated_volume)
                 × (1 + |same_game_eligible_model_bias_pct| / 100)
                 × avg_pred_pct
```

Risk tiers:
- **Low** — `same_game_eligible_bias < 30%` AND `avg_pred > 0` (model confident AND game calibrates OK)
- **High** — `same_game_eligible_bias > 60%` OR no same-game data (game is already badly calibrated)
- **Medium** — everything else

In [ ]:
# Merge gated combo stats with game-level eligible bias
df_risk = df_gated_combos.merge(
    df_game_bias[['target_game_id','eligible_model_bias_pct','eligible_product_bias_pct',
                  'eligible_auction_count','eligible_spend_usd']],
    on='target_game_id', how='left'
)

# Also merge cross-event bias where available
df_risk = df_risk.merge(
    df_event_bias[['target_event_name','eligible_model_bias_pct']].rename(
        columns={'target_event_name': 'target_event',
                 'eligible_model_bias_pct': 'cross_event_model_bias_pct'}
    ),
    on='target_event', how='left'
)

# Proxy risk score
df_risk['game_bias_abs'] = df_risk['eligible_model_bias_pct'].abs().fillna(999)
df_risk['proxy_risk_score'] = (
    np.log10(df_risk['gated_auction_count'].clip(lower=1))
    * (1 + df_risk['game_bias_abs'] / 100)
    * df_risk['avg_pred_pct']
)

# Risk tier
def risk_tier(row):
    if pd.isna(row['eligible_model_bias_pct']):
        return 'Unknown (no same-game data)'
    if abs(row['eligible_model_bias_pct']) > 60:
        return 'High'
    if abs(row['eligible_model_bias_pct']) < 30:
        return 'Low'
    return 'Medium'

df_risk['risk_tier'] = df_risk.apply(risk_tier, axis=1)
df_risk['combo'] = df_risk['target_game_id'].astype(str) + ' / ' + df_risk['target_event']

# Summary by tier
tier_summary = df_risk.groupby('risk_tier').agg(
    combo_count=('combo', 'count'),
    total_gated_auctions=('gated_auction_count', 'sum'),
    total_suppressed_spend_ub=('suppressed_spend_ub_usd', 'sum'),
).reset_index().sort_values('total_gated_auctions', ascending=False)

print('=== Risk Tier Summary ===')
display(tier_summary)

print('\n=== LOW RISK: Top candidates to unlock ===')
low = df_risk[df_risk['risk_tier'] == 'Low'].sort_values('gated_auction_count', ascending=False)
display(low[['combo','gated_auction_count','avg_pred_pct','eligible_model_bias_pct',
             'cross_event_model_bias_pct','suppressed_spend_ub_usd']].head(20))

print('\n=== HIGH RISK: Keep gated or retrain ===')
high = df_risk[df_risk['risk_tier'] == 'High'].sort_values('proxy_risk_score', ascending=False)
display(high[['combo','gated_auction_count','avg_pred_pct','eligible_model_bias_pct',
              'cross_event_model_bias_pct','proxy_risk_score']].head(20))

In [ ]:
color_map = {
    'Low':  '#2ca02c',
    'Medium': '#ff7f0e',
    'High': '#d62728',
    'Unknown (no same-game data)': '#aec7e8',
}

# Drop rows without same-game bias; cast numeric cols to float so plotly handles them cleanly
df_s = df_risk[df_risk['eligible_model_bias_pct'].notna()].copy()
df_s['same_game_bias_abs']  = df_s['eligible_model_bias_pct'].abs().astype(float)
df_s['gated_auction_count'] = df_s['gated_auction_count'].astype(float)
df_s['avg_pred_pct']        = df_s['avg_pred_pct'].astype(float)
df_s['suppressed_spend_ub_usd'] = df_s['suppressed_spend_ub_usd'].fillna(0).astype(float)

fig = px.scatter(
    df_s,
    x='avg_pred_pct',
    y='same_game_bias_abs',
    color='risk_tier',
    color_discrete_map=color_map,
    size='gated_auction_count',
    size_max=40,
    hover_name='combo',
    hover_data=['gated_auction_count','avg_pred_pct','eligible_model_bias_pct',
                'suppressed_spend_ub_usd'],
    title='Gate Removal Risk: Model Confidence vs Same-Game Eligible Bias',
    labels={
        'avg_pred_pct': 'Gated Combo Avg Pred % (model confidence)',
        'same_game_bias_abs': 'Same-Game Eligible |Model Bias| % (calibration quality)',
    },
    template='plotly_white',
    height=600,
)
fig.add_hline(y=30, line_dash='dot', line_color='orange', annotation_text='Low→Medium boundary (30%)')
fig.add_hline(y=60, line_dash='dot', line_color='red',    annotation_text='Medium→High boundary (60%)')
fig.show()

# Separately list unknown-tier combos (no same-game data, excluded from scatter)
unk = df_risk[df_risk['risk_tier'] == 'Unknown (no same-game data)']
if len(unk) > 0:
    print(f'\n{len(unk)} combos with no same-game eligible data (excluded from scatter):')
    display(unk[['combo','gated_auction_count','avg_pred_pct','suppressed_spend_ub_usd']].head(20))

---
## 5. Summary — Gate Removal Risk Assessment

In [ ]:
total     = len(df_risk)
low_n     = (df_risk['risk_tier'] == 'Low').sum()
med_n     = (df_risk['risk_tier'] == 'Medium').sum()
high_n    = (df_risk['risk_tier'] == 'High').sum()
unk_n     = (df_risk['risk_tier'] == 'Unknown (no same-game data)').sum()

low_vol   = df_risk[df_risk['risk_tier'] == 'Low']['gated_auction_count'].sum()
high_vol  = df_risk[df_risk['risk_tier'] == 'High']['gated_auction_count'].sum()
total_vol = df_risk['gated_auction_count'].sum()

low_spend  = df_risk[df_risk['risk_tier'] == 'Low']['suppressed_spend_ub_usd'].sum()
high_spend = df_risk[df_risk['risk_tier'] == 'High']['suppressed_spend_ub_usd'].sum()
total_spend = df_risk['suppressed_spend_ub_usd'].sum()

print('=' * 68)
print('GATE REMOVAL RISK SUMMARY')
print('=' * 68)
print(f'Analysis window  : {ANALYSIS_START} → today (excl downtime)')
print(f'Gated combos     : {total} (game_id, event) pairs with >= 100 auctions')
print(f'Total gated auctions: {total_vol:,}')
print(f'Total suppressed spend UB: ${total_spend:,.0f}')
print()
print('NOTE: Direct bias cannot be computed for gated combos (no outcomes)')
print('Risk is estimated via same-game eligible model bias as a proxy.')
print()
print('RISK BREAKDOWN')
print(f'  Low  risk ({low_n:3d} combos): {low_vol:>12,} auctions | ${low_spend:>12,.0f} suppressed spend UB')
print(f'  Med  risk ({med_n:3d} combos): eligible bias 30–60%')
print(f'  High risk ({high_n:3d} combos): {high_vol:>12,} auctions | ${high_spend:>12,.0f} suppressed spend UB')
print(f'  Unknown   ({unk_n:3d} combos): no same-game eligible data')
print()
print('TOP LOW-RISK UNLOCK CANDIDATES (>= 1000 gated auctions):')
top_low = low[low['gated_auction_count'] >= 1000][['combo','gated_auction_count',
    'avg_pred_pct','eligible_model_bias_pct','suppressed_spend_ub_usd']].head(10)
for _, r in top_low.iterrows():
    print(f'  {r["combo"]:<55} auctions={r["gated_auction_count"]:>8,}  game_bias={r["eligible_model_bias_pct"]:>+7.1f}%')
print('=' * 68)